# P04 · Motor de series temporales financieras
**QuantLab | Módulo 1 · Fundamentos financieros | Práctica 04/12**

**Objetivo:** transformar trades de BTC/EUR en velas OHLCV, retornos, medidas de volatilidad y objetivos futuros, sin introducir información del futuro en las variables explicativas.

**Requisitos:** P01–P03, Python, pandas y operaciones básicas con series. Compatible con Colab, Jupyter y VS Code. Trabajaremos primero con una muestra sintética reproducible y después con Kraken o el CSV de P03. No ejecuta órdenes ni requiere credenciales.

**Ruta didáctica:** 1) datos y unidades → 2) velas → 3) retornos → 4) volatilidad → 5) objetivo futuro → 6) ejercicios y exportación.

Las funciones están en **`src/quantlab/p04/timeseries.py`** y sus pruebas en **`tests/test_timeseries.py`** (`pytest` desde la raíz del proyecto). Los ejercicios están resueltos en la sección 8.

## 1. Configuración
En Colab las bibliotecas suelen estar instaladas. En un entorno local, si faltan, ejecuta `%pip install pandas numpy matplotlib requests` en una celda. Los parámetros se centralizan aquí; ejecuta las celdas en orden.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p04.timeseries import (SYNTHETIC_HIGH_VOL_S, SYNTHETIC_START, add_returns, add_target,
                                     add_volatility, build_ohlcv, compare_frequencies, fetch_trades,
                                     forward_fill_comparison, load_trades_csv, manual_ohlcv,
                                     return_decomposition, synthetic_trades, target_alignment,
                                     validate_trades, volatility_reaction)

DATA_MODE = 'synthetic'  # 'synthetic', 'live' o 'csv'
CSV_PATH = 'quantlab_exports/p03_trades.csv'  # ajusta al nombre exportado en P03
PAIR = 'XBTEUR'
SYMBOL = 'BTC/EUR'
WINDOW = '1min'
VOL_WINDOW = 10  # número de retornos, no segundos
HORIZON = 3      # número de intervalos hacia delante
EXPORT = False
URL = 'https://api.kraken.com/0/public/Trades'
plt.rcParams.update({'figure.figsize': (10, 3.5), 'axes.grid': True,
                     'grid.alpha': .22, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

## 2. Datos: de un evento a una observación
Cada fila contiene una ejecución: `timestamp` (UTC), `price` (EUR/BTC), `quantity` (BTC). Si existe ID de trade se conserva. No eliminamos transacciones solo porque tengan el mismo precio o timestamp: pueden ser ejecuciones distintas.

Fuentes: Kraken Recent Trades y recuperación histórica de trades. Una consulta reciente no garantiza una duración fija ni cobertura histórica completa; por ello las ventanas de los extremos se marcan conservadoramente.

El modo CSV acepta el fichero `*_trades.csv` de P03, no el `*_windows.csv`. Debe contener solo BTC/EUR de un exchange, sin mezclar capturas solapadas. Si existen empates temporales, mantenemos el orden de origen; para reconstrucciones exactas habrá que verificar la secuencia del proveedor.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿los trades tienen sentido?</b>

`validate_trades` se detiene con un error si:

- Faltan las columnas `timestamp`, `price` o `quantity`, o no hay filas.
- Hay timestamps vacíos, o precios o cantidades ≤ 0 o no finitos.
- Hay `trade_id` duplicados: indicaría capturas solapadas.

No elimina trades con el mismo precio o timestamp, y ordena de forma estable para conservar el orden de origen en los empates. El modo `live` reutiliza la descarga de P03 (`fetch_recent_trades`).

</div>

In [ ]:
# 🔎 FASE DE VALIDACIÓN: validate_trades está en src/quantlab/p04/timeseries.py
print(inspect.getsource(validate_trades))

if DATA_MODE == 'synthetic':
    trades, provenance = synthetic_trades(SYMBOL)
elif DATA_MODE == 'live':
    trades, provenance = fetch_trades(PAIR, SYMBOL)  # si falla, lanza el error
elif DATA_MODE == 'csv':
    trades, provenance = load_trades_csv(CSV_PATH, SYMBOL)
else:
    raise ValueError('DATA_MODE no reconocido')
print(provenance)
print(f'Trades: {len(trades):,} | UTC: {trades.timestamp.min()} — {trades.timestamp.max()}')
display(trades.head())

## 3. Construcción de OHLCV
En una ventana $[t-\Delta,t)$, *open* es el primer precio, *high* el máximo, *low* el mínimo y *close* el último. Volumen = suma de cantidades; volumen monetario = $\sum_i p_i q_i$.

Usamos la hora de cierre como etiqueta: una vela etiquetada 10:01 contiene operaciones desde 10:00 inclusive hasta 10:01 exclusive. Sus datos solo se conocen después de cerrar ese intervalo (más la latencia de recepción). En un minuto sin trades observados no inventamos precios: OHLC = NaN y volumen observado = 0. No podemos distinguir ausencia real de trades de un fallo de captura sin metadatos adicionales.

In [ ]:
print(inspect.getsource(build_ohlcv))

bars = build_ohlcv(trades, WINDOW)
display(bars.head(8).round(5))
print('Velas:', len(bars), '| Vacías:', bars.is_empty.sum(), '| Fronteras:', bars.boundary_window.sum())

### 3.1 Comprobar con un ejemplo manual
Con 5 trades a precios 100, 105, 98, 103 y 104, las cantidades 0,2; 0,5; 0,3; 0,4 y 0,6 BTC suman 2 BTC. Debemos obtener O=100, H=105, L=98 y C=104. Añadimos trades antes y después, y un hueco, para comprobar las fronteras temporales.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: vela calculada a mano, fronteras, huecos y conservación de totales
example = pd.DataFrame({'timestamp':pd.to_datetime([
    '2026-01-01 09:59:59','2026-01-01 10:00:01','2026-01-01 10:00:10',
    '2026-01-01 10:00:20','2026-01-01 10:00:40','2026-01-01 10:00:55',
    '2026-01-01 10:01:00','2026-01-01 10:03:00'], utc=True),
    'price':[99,100,105,98,103,104,106,108], 'quantity':[.1,.2,.5,.3,.4,.6,.1,.1]})
example_bars = build_ohlcv(example)
row = example_bars.loc['2026-01-01 10:01:00+00:00']
assert np.allclose(row[['open','high','low','close','volume']].astype(float), [100,105,98,104,2])
assert example_bars.loc['2026-01-01 10:02:00+00:00','open'] == 106
assert example_bars.loc['2026-01-01 10:03:00+00:00','is_empty']
assert pd.isna(example_bars.loc['2026-01-01 10:03:00+00:00','close'])
assert bars.trade_count.sum() == len(trades)
assert np.isclose(bars.volume.sum(), trades.quantity.sum())
assert np.isclose(bars.notional_volume.sum(), (trades.price*trades.quantity).sum())
valid = bars.loc[~bars.is_empty]
assert (valid.high >= valid[['open','close']].max(axis=1)).all()
assert (valid.low <= valid[['open','close']].min(axis=1)).all()
print('9 comprobaciones OHLCV superadas.')
display(example_bars[['open','high','low','close','volume','is_empty','boundary_window']])

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Velas OHLCV</b>

- La vela de las **10:01** contiene los trades de [10:00:00, 10:01:00): O = 100, H = 105, L = 98, C = 104 y 2 BTC. El trade de las 09:59:59 cae en la vela anterior y el de las 10:01:00 en la siguiente.
- La vela de las **10:03** no tiene trades: OHLC = NaN y volumen 0. **No se inventa un precio.**
- La primera y la última vela son de **frontera**: pueden estar incompletas.
- Ninguna vela pierde ni duplica trades: número, volumen y nominal coinciden con los de los trades.

📄 Ver `docs/conceptos/series_temporales.md`

</div>

### 3.2 Visualizar velas y volumen
Dibujamos las últimas 35 ventanas, incluyendo huecos. Las mechas muestran el máximo y mínimo; el cuerpo une apertura y cierre. Verde/azul indica cierre mayor o igual a apertura, naranja indica caída. El volumen observado mide cantidad ejecutada, no liquidez pendiente.

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator

def plot_candles(df, count=35):
    d = df.tail(count)
    fig, ax = plt.subplots(figsize=(10,4))
    for i, (_, r) in enumerate(d.iterrows()):
        if r.is_empty:
            ax.axvspan(i-.35,i+.35,color='gray',alpha=.16)
            continue
        color = '#26758b' if r.close >= r.open else '#b36a37'
        ax.vlines(i,r.low,r.high,color=color,linewidth=1.3)
        if r.open == r.close:
            ax.hlines(r.open,i-.28,i+.28,color=color,linewidth=2)
        else:
            ax.add_patch(Rectangle((i-.28,min(r.open,r.close)),.56,abs(r.close-r.open),color=color))
    ax.autoscale_view()
    ax.set_xlim(-.7,len(d)-.3)
    ticks=np.unique(np.linspace(0,len(d)-1,min(7,len(d)),dtype=int))
    ax.set_xticks(ticks, [d.index[i].strftime('%H:%M:%S') for i in ticks],rotation=25)
    ax.set(title=f'{DATA_MODE} · BTC/EUR | Velas {WINDOW}', xlabel='Cierre UTC', ylabel='EUR/BTC')
    fig.tight_layout(); plt.show()

plot_candles(bars)
fig, ax = plt.subplots()
ax.bar(bars.index, bars.volume, width=pd.Timedelta(WINDOW)/pd.Timedelta('1D')*.8, color='#26758b')
ax.set(title=f'{DATA_MODE} · Volumen observado por {WINDOW}', xlabel='Cierre UTC',ylabel='BTC')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 4. Retornos: transformar precios en variaciones comparables
Retorno simple: $r_t=P_t/P_{t-1}-1$. Retorno logarítmico: $\ell_t=\ln(P_t/P_{t-1})$. Trabajamos internamente en decimales: 0,01 = 1 % = 100 bps.

Los precios de cierre de barras vacías o de frontera no se utilizan para calcular retornos de horizonte fijo. No rellenamos huecos automáticamente. Un retorno que cruza un hueco no es comparable con un retorno de un minuto. La rentabilidad acumulada se reinicia por segmento de precios válidos y es relativa al primer cierre de ese segmento.

In [ ]:
print(inspect.getsource(add_returns))

series = add_returns(bars)
# 🔎 FASE DE VALIDACIÓN: ln(1 + r) = ℓ, y no hay retornos desde ni hacia cierres excluidos
assert np.allclose(np.log1p(series.simple_return.dropna()),series.log_return.dropna())
assert series.loc[series.analysis_close.isna(),'simple_return'].isna().all()
assert series.loc[series.analysis_close.shift(1).isna(),'simple_return'].isna().all()
print('3 comprobaciones de retornos superadas.')
display(series[['close','analysis_close','simple_return','log_return','cumulative_return']].head(8).round(7))
fig, ax = plt.subplots()
ax.plot(series.index, series.log_return*10000, color='#32658a', linewidth=1)
ax.axhline(0,color='gray',linewidth=.8)
ax.set(title=f'{DATA_MODE} · Retornos logarítmicos por {WINDOW}',ylabel='Puntos básicos',xlabel='Cierre UTC')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 5. Volatilidad móvil y volatilidad realizada
La volatilidad móvil es la desviación típica muestral de los últimos $N$ retornos logarítmicos: $s_t=\sqrt{\sum (\ell_i-\bar\ell)^2/(N-1)}$. Su unidad es retorno por intervalo. No indica dirección.

La volatilidad realizada de una ventana, definida aquí como $\sqrt{\sum \ell_i^2}$, corresponde a todo el horizonte de $N$ intervalos. No es la misma medida. No anualizamos: hacerlo requeriría supuestos adicionales sobre frecuencia, cobertura y dependencia temporal. Exigimos N retornos consecutivos válidos; si falta uno, el resultado queda vacío.

In [ ]:
print(inspect.getsource(add_volatility))

series = add_volatility(series,VOL_WINDOW)
# 🔎 FASE DE VALIDACIÓN: cálculo a mano y ventanas incompletas vacías
probe = pd.DataFrame({'log_return':[.01,-.01,.01,np.nan,.02]})
vol = add_volatility(probe,3)
assert np.isclose(vol.loc[2,'rolling_volatility'],np.std([.01,-.01,.01],ddof=1))
assert np.isclose(vol.loc[2,'realized_volatility'],np.sqrt(3*.01**2))
assert vol.loc[[0,1,3,4],'rolling_volatility'].isna().all()
print('3 comprobaciones de volatilidad superadas.')
fig, ax = plt.subplots()
ax.plot(series.index,series.rolling_volatility*10000,color='#32658a',label=f'Desv. típica por {WINDOW} (N={VOL_WINDOW})')
ax.set(title=f'{DATA_MODE} · Volatilidad móvil de retornos',ylabel='bps por intervalo',xlabel='Cierre UTC')
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 6. Retorno futuro: preparar el objetivo de investigación
Si decidimos en el cierre de la vela $t$, el retorno futuro de $h$ intervalos es $y_t=P_{t+h}/P_t-1$.

La variable objetivo no es una feature. Solo podemos conocerla después de que transcurra el horizonte. Aquí exigimos que todos los precios intermedios estén disponibles y fuera de las fronteras. Los últimos h registros no tienen objetivo conocido. Este retorno close-to-close es una etiqueta de investigación, no un P&L ejecutable: desconocemos bid, ask, comisiones, profundidad y latencia. No se debe incluir en las variables de entrada ni usar antes de que sea observable.

In [ ]:
print(inspect.getsource(add_target))

series = add_target(series,HORIZON)
# 🔎 FASE DE VALIDACIÓN: el objetivo nunca cruza huecos ni fronteras, y no entra en las features
assert series.forward_return.tail(HORIZON).isna().all()
assert series.loc[series.analysis_close.isna(),'forward_return'].isna().all()
for step in range(1,HORIZON+1):
    assert series.loc[series.analysis_close.shift(-step).isna(),'forward_return'].isna().all()
check = pd.DataFrame({'analysis_close':[100.,110.,121.,np.nan,100.,120.],
                      'segment_id':pd.array([0,0,0,None,1,1],dtype='Int64')},
                     index=pd.date_range('2026-01-01',periods=6,freq='min',tz='UTC'))
check = add_target(check,2)
assert np.isclose(check.forward_return.iloc[0],.21)
assert check.forward_return.iloc[1:].isna().all()
print('Comprobaciones de alineación temporal superadas.')
feature_columns = ['simple_return','log_return','rolling_volatility','realized_volatility','price_range','volume','trade_count']
assert 'forward_return' not in feature_columns
research_table = series[feature_columns+['forward_return','target_end_utc']].dropna()
print('Filas completas para una investigación posterior:',len(research_table))
display(research_table.head().round({c: 7 for c in feature_columns + ['forward_return']}))

## 7. Comparar frecuencias temporales
La resolución cambia la cantidad de observaciones, los huecos y la escala de volatilidad. No compares desviaciones típicas de 5 segundos y 5 minutos como si midieran lo mismo. La suma de volumen debe mantenerse si incluimos todas las ventanas de la misma captura, incluidas las fronteras.

In [ ]:
print(inspect.getsource(compare_frequencies))

comparison = compare_frequencies(trades, ['5s','15s','1min','5min'])
assert np.allclose(comparison.total_volume_btc,trades.quantity.sum())
display(comparison.round(5))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Frecuencias (muestra sintética)</b>

| Frecuencia | Velas | Vacías | Retornos válidos | Desv. típica por vela |
|---|---|---|---|---|
| 5 s | 1.079 | **166** | 789 | 0,91 bps |
| 15 s | 360 | 8 | 348 | 1,41 bps |
| 1 min | 90 | 2 | 84 | 3,10 bps |
| 5 min | 18 | 0 | 15 | 5,62 bps |

- El volumen total (127,90 BTC) es **el mismo** en todas las frecuencias: solo cambia cómo se reparte.
- Cuanto más fina la resolución, **más velas vacías**: con 5 s, una de cada seis no tiene trades.
- La desviación típica **crece con el intervalo**: 0,91 bps por vela de 5 s frente a 5,62 bps por vela de 5 min. Mide "bps por vela", así que cada cifra solo es comparable con otras de la misma frecuencia.
- Con 5 min solo quedan 15 retornos: muy pocos para estimar nada con fiabilidad.

</div>

## 8. Ejercicios
**E1 · OHLCV manual.** Sin llamar a `build_ohlcv`, implementa una función que calcule O, H, L, C y volumen de un grupo de trades. Verifica con los cinco del ejemplo (100, 105, 98, 104, 2 BTC).

**E2 · Retornos.** Calcula retornos simples y logarítmicos sobre precios 100 → 110 → 100. Comprueba que el retorno acumulado compuesto es cero aunque la suma de retornos simples no lo sea. Pista: `np.prod(1+r)-1`.

**E3 · Volatilidad.** Calcula volatilidad con ventanas de 5, 10 y 20 retornos. ¿Qué ventana reacciona antes al cambio de régimen introducido en la simulación? Identifica la unidad y explica por qué no predice la dirección.

**E4 · Huecos y fronteras.** Localiza un intervalo vacío. Explica qué ocurriría si rellenases su precio con el último cierre conocido; compara el número de retornos válidos, sin convertir esa imputación en un hecho de mercado.

**E5 · Alineación temporal.** Construye objetivos de 1, 3 y 5 ventanas. Inspecciona manualmente tres filas; demuestra que su fecha objetivo es posterior a su fecha de observación y que no cruza huecos. ¿Qué pasaría al introducir el objetivo como feature?

**E6 · Datos reales.** Elige `DATA_MODE='live'` o importa el CSV original de P03. Ejecuta todo de nuevo. Describe cobertura, resolución, número de huecos, volatilidad y diferencias frente a la muestra sintética. Si hay pocos retornos, elige una frecuencia razonable o una captura más larga; no inventes datos.

### E1 · OHLCV manual

In [ ]:
print(inspect.getsource(manual_ohlcv))

five = example.iloc[1:6]  # los cinco trades de [10:00, 10:01)
e1 = manual_ohlcv(five)
display(e1.to_frame('E1 manual'))
# 🔎 FASE DE VALIDACIÓN: coincide con el cálculo a mano y con build_ohlcv
assert list(e1) == [100, 105, 98, 104, 2]
assert np.allclose(e1, example_bars.loc['2026-01-01 10:01:00+00:00', ['open','high','low','close','volume']].astype(float))
print('E1 verificado: O=100, H=105, L=98, C=104, volumen = 2 BTC.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Una vela es un resumen ordenado en el tiempo</b>

- **Open** y **close** dependen del **orden** de los trades: son el primero y el último. Por eso la función ordena por timestamp antes de calcular (de forma estable, para respetar los empates).
- **High** y **low** no dependen del orden: son el máximo y el mínimo.
- El **volumen** suma las cantidades (0,2 + 0,5 + 0,3 + 0,4 + 0,6 = 2 BTC), no los precios.
- Un grupo vacío devuelve OHLC = NaN y volumen 0, igual que `build_ohlcv`.

</div>

### E2 · Retornos simples, logarítmicos y compuestos

In [ ]:
print(inspect.getsource(return_decomposition))

e2 = return_decomposition([100., 110., 100.])
display(e2.round(6))
display(pd.Series(e2.attrs, name='Total').to_frame().round(6))
# 🔎 FASE DE VALIDACIÓN: el compuesto es 0 aunque la suma de simples no lo sea
assert np.isclose(e2.attrs['compounded'], 0) and e2.attrs['sum_simple'] > 0
assert np.isclose(e2.attrs['sum_log'], 0)
print('E2 verificado: compuesto = 0, suma de simples = +0,909 %, suma de logarítmicos = 0.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Los retornos simples se componen; los logarítmicos se suman</b>

| Tramo | Retorno simple | Retorno logarítmico |
|---|---|---|
| 100 → 110 | +10,000 % | +9,531 % |
| 110 → 100 | −9,091 % | −9,531 % |
| **Suma** | **+0,909 %** ❌ | **0** ✅ |
| **Compuesto** ∏(1 + r) − 1 | **0** ✅ | — |

- El precio vuelve a 100, así que el resultado real es **0**. La suma de simples dice +0,909 %: es un error, porque la caída se aplica sobre una base mayor (110).
- Los retornos **simples** se agregan **multiplicando**: (1 + 0,10) × (1 − 0,0909) − 1 = 0.
- Los retornos **logarítmicos** se agregan **sumando**: ln(110/100) + ln(100/110) = 0. Es su gran ventaja para series temporales.
- Para retornos pequeños (pocos bps), simple y logarítmico casi coinciden. Para movimientos grandes, no.

</div>

### E3 · Volatilidad con ventanas de 5, 10 y 20 retornos

In [ ]:
print(inspect.getsource(volatility_reaction))

fig, ax = plt.subplots()
for n, color in [(5, '#b36a37'), (10, '#32658a'), (20, '#26816d')]:
    ax.plot(series.index, add_volatility(series, n).rolling_volatility * 10_000, color=color, label=f'N = {n}')
if provenance['mode'] == 'synthetic':
    # Primera vela completa del tramo de alta volatilidad de la simulación
    regime_start = SYNTHETIC_START + pd.Timedelta(seconds=SYNTHETIC_HIGH_VOL_S[0]) + pd.Timedelta(WINDOW)
    regime_end = SYNTHETIC_START + pd.Timedelta(seconds=SYNTHETIC_HIGH_VOL_S[1])
    ax.axvspan(regime_start, regime_end, color='gray', alpha=.15, label='Régimen de alta volatilidad')
    display(volatility_reaction(series, regime_start, (5, 10, 20), factor=2.0))
ax.set(title=f'{DATA_MODE} · Volatilidad móvil por {WINDOW}', ylabel='bps por intervalo', xlabel='Cierre UTC')
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · La ventana corta reacciona antes, pero es más ruidosa</b>

El régimen de alta volatilidad de la simulación empieza con la vela de las **10:49**. Alerta = primera vela en que la volatilidad supera el doble de su mediana anterior:

| N | Mediana antes del cambio | Primera alerta | Velas de retraso | Máximo | Valores válidos |
|---|---|---|---|---|---|
| **5** | 1,79 bps | **10:54** | **5** | 9,01 bps | 76 |
| 10 | 1,88 bps | 10:56 | 7 | 7,33 bps | 66 |
| 20 | 1,80 bps | 10:58 | 9 | 5,53 bps | 46 |

- **La ventana de 5 retornos reacciona antes**: 4 velas antes que la de 20. Cada retorno nuevo pesa 1/5 en lugar de 1/20.
- **El precio de reaccionar rápido es el ruido**: la de 5 sube y baja más con retornos aislados. La de 20 es más estable, pero llega tarde y pierde más valores al principio y tras el hueco (46 válidos frente a 76).
- **Unidad:** desviación típica de retornos logarítmicos **por intervalo de 1 minuto**, expresada en bps. No está anualizada.
- **No predice la dirección** porque eleva al cuadrado las desviaciones: una subida y una bajada del mismo tamaño aportan lo mismo. Mide **cuánto** se mueve el precio, no **hacia dónde**.

</div>

### E4 · Huecos y fronteras

In [ ]:
print(inspect.getsource(forward_fill_comparison))

gaps = series.loc[series.is_empty, ['close', 'volume', 'trade_count', 'analysis_close']]
print('Velas vacías:'); display(gaps)
if len(gaps):
    first_gap = gaps.index[0]
    around = series.index.get_loc(first_gap)
    window_rows = series.iloc[max(around - 2, 0): around + len(gaps) + 2]
    filled = window_rows.close.ffill()
    display(pd.DataFrame({'close real': window_rows.close, 'close imputado (ffill)': filled,
                          'retorno real (bps)': window_rows.log_return * 1e4,
                          'retorno imputado (bps)': np.log(filled / filled.shift(1)) * 1e4}).round(3))
e4 = forward_fill_comparison(bars)
display(e4.round(3))
print('Retornos creados por la imputación en huecos o a su salida:', e4.attrs['imputed_returns'])

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · Rellenar un hueco inventa retornos que no existieron</b>

En la simulación, las velas de las **10:31** y **10:32** están vacías (no hubo trades entre 10:30 y 10:32).

| | Retornos válidos | Retornos exactamente 0 | Desv. típica |
|---|---|---|---|
| Sin imputar | 84 | 0 | 3,10 bps |
| Rellenando con el último cierre | **87** | **2** | 3,05 bps |

Si rellenamos los huecos con el último cierre conocido (59.941,17 €):

- Aparecen **2 retornos exactamente 0** (10:31 y 10:32). Dicen "el precio no se movió", pero en realidad **no sabemos** qué pasó: no hubo trades observados.
- El retorno de las **10:33** pasa a medir el cambio desde las 10:30: **abarca 3 minutos**, pero se trata como si fuera de 1 minuto.
- La volatilidad **baja artificialmente** (de 3,10 a 3,05 bps), porque se añaden ceros falsos.
- Los objetivos futuros podrían cruzar el hueco como si hubiera continuidad.
- Ganamos 3 "retornos válidos" que **no son hechos de mercado**: son un supuesto. Si se imputa, hay que declararlo y marcar esas filas.

</div>

### E5 · Alineación temporal de objetivos de 1, 3 y 5 ventanas

In [ ]:
print(inspect.getsource(target_alignment))

e5 = target_alignment(series, (1, 3, 5))
display(e5)
assert e5[['all_after_observation', 'exact_horizon', 'no_gap_crossed']].all().all()

# Inspección manual de tres filas con h = 3: una al principio, una justo antes del hueco y una justo después
t3 = add_target(series, 3)
known = t3.loc[t3.forward_return.notna()]
gap_start = series.index[series.is_empty][0] if series.is_empty.any() else known.index[len(known) // 2]
rows = [known.index[0], known.index[known.index < gap_start][-1], known.index[known.index > gap_start][0]]
manual = t3.loc[rows, ['analysis_close', 'segment_id', 'target_end_utc', 'forward_return']].copy()
manual['close_at_target'] = [t3.loc[end, 'analysis_close'] for end in manual.target_end_utc]
manual['check_y'] = manual.close_at_target / manual.analysis_close - 1
manual['segment_at_target'] = [t3.loc[end, 'segment_id'] for end in manual.target_end_utc]
display(manual)
assert (manual.target_end_utc > manual.index).all()
assert np.allclose(manual.check_y, manual.forward_return)
assert (manual.segment_id == manual.segment_at_target).all()
print('Tres filas inspeccionadas: objetivo posterior, mismo segmento y retorno recalculado a mano.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · El objetivo es siempre posterior y nunca cruza un hueco</b>

| h | Objetivos conocidos | Objetivo posterior | Exactamente h velas | Sin cruzar huecos | Sin objetivo al final |
|---|---|---|---|---|---|
| 1 | 84 | ✅ | ✅ | ✅ | 1 |
| 3 | 80 | ✅ | ✅ | ✅ | 3 |
| 5 | 76 | ✅ | ✅ | ✅ | 5 |

Inspección manual con h = 3 (simulación):

| Observación | Cierre | Objetivo | Segmento | Comprobación |
|---|---|---|---|---|
| 10:02 | 59.989,85 € | 10:05 | 1 → 1 | 3 min después, mismo segmento |
| 10:27 | 59.915,87 € | 10:30 | 1 → 1 | El último objetivo antes del hueco termina a las 10:30 |
| 10:33 | 59.946,77 € | 10:36 | 3 → 3 | El primero después del hueco empieza en el segmento nuevo |

- Las filas de las 10:28, 10:29 y 10:30 **no tienen objetivo**: su horizonte cruzaría el hueco.
- Cuanto mayor es h, **menos objetivos** se conocen: se pierden h filas al final y alrededor de cada hueco.
- **Si metiéramos el objetivo como feature** (*look-ahead*), el modelo vería el futuro. En el backtest parecería casi perfecto, porque estaría "prediciendo" con la respuesta en la mano. En tiempo real esa información no existe en el momento de decidir, y el resultado se desplomaría. Por eso `forward_return` nunca está en `feature_columns`.

</div>

### E6 · Datos reales de Kraken
Cambia `LIVE_E6` a `True` para descargar trades recientes y repetir todo el proceso. Si la conexión falla, la celda se detiene con el error: no hay sustitución por datos ficticios. Para usar el CSV de P03, cambia `DATA_MODE` en la configuración y vuelve a ejecutar desde el principio.

In [ ]:
LIVE_E6 = False

if LIVE_E6:
    live_trades, live_prov = fetch_trades(PAIR, SYMBOL)  # si falla, lanza el error
    print('Recepción UTC:', live_prov['received_at'])
    print(f'Trades: {len(live_trades):,} | UTC: {live_trades.timestamp.min()} — {live_trades.timestamp.max()}')
    live_bars = build_ohlcv(live_trades, WINDOW)
    live_series = add_target(add_volatility(add_returns(live_bars), VOL_WINDOW), HORIZON)
    summary = pd.DataFrame({
        'Sintética': [len(trades), len(bars), int(bars.is_empty.sum()), int(series.log_return.count()),
                      series.log_return.std() * 1e4, series.rolling_volatility.median() * 1e4,
                      len(series[feature_columns + ['forward_return']].dropna())],
        'Kraken': [len(live_trades), len(live_bars), int(live_bars.is_empty.sum()), int(live_series.log_return.count()),
                   live_series.log_return.std() * 1e4, live_series.rolling_volatility.median() * 1e4,
                   len(live_series[feature_columns + ['forward_return']].dropna())],
    }, index=['Trades', f'Velas de {WINDOW}', 'Velas vacías', 'Retornos válidos',
              'Desv. típica (bps por vela)', f'Volatilidad móvil mediana N={VOL_WINDOW} (bps)', 'Filas de investigación'])
    display(summary.round(3))
    display(compare_frequencies(live_trades).round(4))
    plot_candles(live_bars)
else:
    print('LIVE_E6 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Kraken BTC/EUR (19-11-2026, recepción 18:53:47 UTC)</b>

**Cobertura:** 1.000 trades, de 17:53:40 a 18:53:45 UTC (**60 minutos**). Es lo que devuelve una sola consulta REST, no una duración elegida.

| | Sintética | Kraken |
|---|---|---|
| Trades | 1.755 | 1.000 |
| Velas de 1 min | 90 | 61 |
| Velas vacías (1 min) | 2 | **0** |
| Retornos válidos | 84 | 58 |
| Desv. típica de retornos de 1 min | 3,10 bps | **8,64 bps** |
| Volatilidad móvil mediana (N = 10) | 1,88 bps | **8,24 bps** |
| Filas completas de investigación | 60 | 46 |

Por frecuencia (Kraken):

| Frecuencia | Velas | Vacías | Retornos válidos | Desv. típica por vela |
|---|---|---|---|---|
| 5 s | 722 | **314** | 235 | 2,57 bps |
| 15 s | 242 | 23 | 197 | 4,41 bps |
| 1 min | 61 | 0 | 58 | 8,64 bps |
| 5 min | 13 | 0 | 10 | 21,96 bps |

- **Resolución razonable: 1 minuto.** Con 5 s, el 43 % de las velas están vacías y solo quedan 235 retornos en segmentos cortos. Con 5 min solo hay 10 retornos.
- **El mercado real fue unas 4 veces más volátil** que la simulación en reposo (8,2 frente a 1,9 bps por minuto), comparable a su tramo de alta volatilidad.
- **No hubo huecos de 1 minuto**, pero sí muchos de 5 s: el mercado real negocia a ráfagas (mediana de 13 trades por minuto, máximo 54).
- **Pocos datos:** 46 filas de investigación no alcanzan para estudiar nada. Hace falta una captura continua y mucho más larga; no se pueden inventar datos.

</div>

## 9. Exportación y siguientes pasos
Exportamos las velas y las variables calculadas por separado de la fuente original. Conservamos los huecos y los metadatos, incluidos frecuencia, horizonte y política de exclusión. En un proyecto profesional guardaríamos también el fichero bruto inmutable y las versiones del código. No guardes grandes ficheros de mercado en Git.

In [ ]:
if EXPORT:
    folder=Path('quantlab_exports'); folder.mkdir(exist_ok=True)
    stamp=pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    prefix=folder/f'p04_{DATA_MODE}_{stamp}'
    series.to_csv(str(prefix)+'_series.csv')
    metadata={**provenance,'window':WINDOW,'vol_window':VOL_WINDOW,'horizon':HORIZON,
              'bar_label':'right','closed':'left','timezone':'UTC',
              'first_last_bars':'excluded_from_returns','gaps':'no_fill',
              'source_start':trades.timestamp.min().isoformat(),'source_end':trades.timestamp.max().isoformat(),
              'feature_columns':feature_columns}
    Path(str(prefix)+'_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    print('Exportado:',prefix)
else:
    print('Exportación desactivada. Activa EXPORT para guardar los resultados.')

## 10. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P04</b>

**Qué significa una vela y cuándo se conoce su información.**
Una vela resume los trades de un intervalo [t − Δ, t): primer precio (open), máximo (high), mínimo (low), último (close), volumen y número de trades. Se etiqueta con su cierre y **solo se conoce cuando ese intervalo termina**, más la latencia de recepción. Usarla antes es mirar el futuro.

**Diferencias entre retorno simple, logarítmico y acumulado.**
El simple (P_t / P_{t−1} − 1) es el cambio porcentual real y se agrega multiplicando. El logarítmico (ln(P_t / P_{t−1})) se agrega sumando y es simétrico (+10 % y −9,09 % dan +0,0953 y −0,0953). El acumulado mide el resultado desde el inicio de un segmento: hay que componer, nunca sumar simples (100 → 110 → 100 suma +0,909 %, pero el resultado real es 0).

**Por qué no se mezclan retornos de diferentes horizontes sin ajuste.**
Su escala es distinta: la desviación típica pasó de 0,91 bps por vela de 5 s a 5,62 bps por vela de 5 min. Un retorno que cruza un hueco abarca varios intervalos y no es comparable con uno de un solo intervalo.

**Qué significa una volatilidad alta y qué no significa.**
Significa que el precio se mueve mucho por intervalo, en cualquier dirección. **No** significa que vaya a subir o bajar, ni que el riesgo futuro sea alto: es una medida del pasado reciente, que depende de la ventana N y de la frecuencia.

**Qué consecuencias tienen los huecos para una investigación.**
Reducen las observaciones válidas, cortan los segmentos y obligan a decidir qué hacer. Rellenarlos inventa retornos cero, alarga retornos a varios intervalos y baja la volatilidad artificialmente. Además, sin metadatos no se distingue un mercado sin trades de un fallo de captura.

**Cómo construir una etiqueta futura sin contaminar las features.**
Calcular el objetivo (P_{t+h} / P_t − 1) solo cuando los h cierres siguientes están en el mismo segmento válido, guardar la fecha en que se conoce (`target_end_utc`), mantenerlo **fuera** de la lista de features y no usarlo nunca antes de esa fecha.

**P05: Momentum y Mean Reversion.** Utilizaremos estas series para crear señales candidatas, medir su relación con los retornos futuros y distinguir asociación de ventaja neta. El objetivo de P04 es construir datos fiables, no encontrar una estrategia rentable.

📄 Resumen en `docs/conclusiones/P04_time_series.md`

</div>